# GNN graph connectivity: train vs validation (and checkpoints)

Computes **Cα proximity graph** statistics using the same edge rules as training (`gnn.data_utils.compute_edges`): sequential edges plus spatial edges within `distance_threshold`.

- **Train / val** indices match `scripts/run_gnn_train_final_models.py` (`StratifiedShuffleSplit` with `val_size`, `seed`).
- **Optional** cluster holdout (`GroupKFold`) when `cluster_id` is present in the geometric CSV.
- **Trained models**: graph connectivity does **not** depend on weights; checkpoint `*_gnn_meta.json` files are listed so you can align runs with the same PDB/threshold settings.

Run from project `notebooks/` (or repo root). `ROOT` resolves to `sequence_to_svm_minimal/`. Requires `jupyter`, `matplotlib` (install if missing).

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedShuffleSplit, GroupKFold

here = Path.cwd().resolve()
repo = here.parent if here.name == "notebooks" else here
ROOT = repo / "sequence_to_svm_minimal"
if not (ROOT / "configs" / "gnn_final_train.json").is_file():
    ROOT = Path("..").resolve()  # legacy: opened from sequence_to_svm_minimal/notebooks
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from configs.load_config import load_gnn_final_train_bundle
from gnn.data_utils import parse_pdb, compute_edges, resolve_peptide_pdb_path
from gnn.checkpoint_meta import load_peptide_gnn_meta, sidecar_meta_path

CONFIG_PATH = ROOT / "configs" / "gnn_final_train.json"
training, feature_sets, architectures, node_groups_cfg, train_feature_sets = load_gnn_final_train_bundle(
    str(CONFIG_PATH)
)

SEED = int(training.get("seed", 42))
VAL_SIZE = 0.2
DISTANCE_THRESHOLD = float(training.get("distance_threshold", 8.0))

csv_path = Path(training["csv_path"])
pdb_dir = Path(training["pdb_dir"])
print("CSV:", csv_path)
print("PDB dir:", pdb_dir)
print("distance_threshold:", DISTANCE_THRESHOLD, "| seed:", SEED, "| val_size:", VAL_SIZE)

In [ ]:
def connectivity_stats(ca_coords: np.ndarray, edge_index, edge_attr, n: int) -> dict:
    """Directed edges from compute_edges; edge_attr[:,2] 0=sequential, 1=spatial."""
    E = int(edge_index.shape[1])
    pairs = {tuple(sorted((int(edge_index[0, k]), int(edge_index[1, k])))) for k in range(E)}
    Eu = len(pairs)
    max_u = n * (n - 1) / 2 if n > 1 else 1.0
    max_d = n * (n - 1) if n > 1 else 1.0
    rho_u = Eu / max_u
    rho_d = E / max_d
    seq_d = int((edge_attr[:, 2] < 0.5).sum().item()) if edge_attr is not None and edge_attr.numel() else 0
    spa_d = int((edge_attr[:, 2] >= 0.5).sum().item()) if edge_attr is not None and edge_attr.numel() else 0
    mean_deg = E / n if n else 0.0
    if spa_d > 0 and edge_attr is not None:
        mask = edge_attr[:, 2] >= 0.5
        mean_spa_dist_norm = float(edge_attr[mask, 0].mean().item())
    else:
        mean_spa_dist_norm = float("nan")
    return {
        "n_residues": n,
        "E_directed": E,
        "E_undirected": Eu,
        "rho_undirected": rho_u,
        "rho_directed": rho_d,
        "n_seq_edges_dir": seq_d,
        "n_spa_edges_dir": spa_d,
        "mean_directed_degree": mean_deg,
        "mean_spatial_edge_dist_norm": mean_spa_dist_norm,
    }


def stats_for_pdb_row(row: pd.Series, pdb_dir: Path, distance_threshold: float) -> dict | None:
    pdb_file = row.get("pdb_file", None)
    pdb_path = resolve_peptide_pdb_path(pdb_dir, pdb_file, row["peptide_id"])
    if pdb_path is None:
        return None
    _, ca_coords, _ = parse_pdb(str(pdb_path))
    n = len(ca_coords)
    if n < 2:
        return None
    edge_index, edge_attr = compute_edges(ca_coords, distance_threshold)
    out = connectivity_stats(ca_coords, edge_index, edge_attr, n)
    out["peptide_id"] = row["peptide_id"]
    out["label"] = int(row["label"])
    if "cluster_id" in row.index and pd.notna(row["cluster_id"]):
        out["cluster_id"] = int(row["cluster_id"])
    return out

In [ ]:
df = pd.read_csv(csv_path)
print(df.shape)
df.head(2)

In [ ]:
rows = []
missing = []
for _, row in df.iterrows():
    s = stats_for_pdb_row(row, pdb_dir, DISTANCE_THRESHOLD)
    if s is None:
        missing.append(row.get("peptide_id", ""))
    else:
        rows.append(s)

conn = pd.DataFrame(rows)
print("graphs:", len(conn), "| missing PDB:", len(missing))
if missing[:5]:
    print("example missing:", missing[:5])
conn.describe()

In [ ]:
labels = np.where(df["label"].values == 1, 1, 0)
splitter = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(splitter.split(np.arange(len(df)), labels))

train_ids = set(df.iloc[train_idx]["peptide_id"].astype(str))
val_ids = set(df.iloc[val_idx]["peptide_id"].astype(str))
conn["split_final_train"] = conn["peptide_id"].astype(str).map(
    lambda pid: "train" if pid in train_ids else ("val" if pid in val_ids else "unknown")
)
n_unknown = (conn["split_final_train"] == "unknown").sum()
if n_unknown:
    print(f"Warning: {n_unknown} graphs not in train/val CSV rows (missing PDBs?).")
conn["split_final_train"].value_counts()

In [ ]:
metric = "rho_undirected"
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, col in zip(axes, ["train", "val"]):
    sub = conn.loc[conn["split_final_train"] == col, metric]
    ax.hist(sub, bins=40, alpha=0.75, color="#2c7fb8" if col == "train" else "#f18f01")
    ax.set_title(f"{col} (n={len(sub)})")
    ax.set_xlabel(metric)
    ax.set_ylabel("count")
fig.suptitle("Graph connectivity vs split (same rule as run_gnn_train_final_models)")
fig.tight_layout()
plt.show()

summ = conn.groupby("split_final_train")[["rho_undirected", "E_undirected", "n_residues", "mean_directed_degree"]].agg(
    ["mean", "median", "std"]
)
summ

In [ ]:
try:
    from scipy.stats import mannwhitneyu

    a = conn.loc[conn["split_final_train"] == "train", "rho_undirected"]
    b = conn.loc[conn["split_final_train"] == "val", "rho_undirected"]
    stat, p = mannwhitneyu(a, b, alternative="two-sided")
    print(f"Mann–Whitney U on rho_undirected (train vs val): statistic={stat:.4g}, p={p:.4g}")
except ImportError:
    print("scipy not installed; skip Mann–Whitney test")

## Optional: cluster holdout (first `GroupKFold` fold)

If `cluster_id` exists, peptides in the same cluster stay in either train or test for that fold. This is **not** the same split as `run_gnn_train_final_models.py` (which uses stratified shuffle only).

In [ ]:
if "cluster_id" in df.columns:
    clusters = df["cluster_id"].values
    gkf = GroupKFold(n_splits=5)
    train_c_idx, test_c_idx = next(gkf.split(np.arange(len(df)), labels, groups=clusters))
    train_c_ids = set(df.iloc[train_c_idx]["peptide_id"].astype(str))
    test_c_ids = set(df.iloc[test_c_idx]["peptide_id"].astype(str))
    conn["split_cluster_fold0"] = conn["peptide_id"].astype(str).map(
        lambda pid: "gkf_train" if pid in train_c_ids else "gkf_test"
    )
    display(conn.groupby("split_cluster_fold0")["rho_undirected"].describe())
else:
    print("No cluster_id column; skip GroupKFold section.")

## Trained checkpoints (metadata only)

Graph topology for a PDB is identical for every model; meta JSON confirms architecture and input layout. Adjust `CHECKPOINT_GLOB` to your `results/gnn/ready_models/run_*` or `checkpoints/latest`.

In [ ]:
CHECKPOINT_GLOB = str(ROOT / "results" / "gnn" / "ready_models" / "run_*" / "*_ready_*.pt")

from glob import glob

ckpts = sorted(glob(CHECKPOINT_GLOB))
meta_rows = []
for p in ckpts:
    mp = sidecar_meta_path(p)
    meta = load_peptide_gnn_meta(p)
    meta_rows.append(
        {
            "checkpoint": p,
            "meta_path": str(mp),
            "has_meta": meta is not None,
            "architecture": (meta or {}).get("architecture"),
            "geo_feature_dim": (meta or {}).get("geo_feature_dim"),
            "esm2_raw_dim": (meta or {}).get("esm2_raw_dim"),
        }
    )

meta_df = pd.DataFrame(meta_rows)
if len(meta_df):
    display(meta_df)
else:
    print("No checkpoints matched:", CHECKPOINT_GLOB)
    print("Train a model or set CHECKPOINT_GLOB to your .pt paths.")